# PubMed-RAG-TR: RAG evaluation and BEIR-style benchmark

Expensive or billed sections are disabled by default. Every result written by a new run goes to `outputs/`.

## Runtime and credentials

The audit cells need only a CPU. Full RAG and BEIR runs should use a CUDA GPU (the three released models total several GB). Generation reads `GOOGLE_API_KEY`; RAGAS reads `OPENAI_API_KEY`; gated Hugging Face models may require `HF_TOKEN` after their access conditions have been accepted. Secrets are never embedded in output records.

The baseline ColBERT stage uses `ragatouille`, which has a narrower dependency range than the rest of the notebook. Install it only when running the Base RAG system: `pip install ragatouille==0.0.9.post2`.

In [ ]:
%pip install -q "datasets==3.6.0" "pandas==2.2.3" "numpy==1.26.4" "scipy==1.15.3" "statsmodels==0.14.4" "rank-bm25==0.2.2" "sentence-transformers==4.1.0" "transformers==4.51.3" "langchain-text-splitters==0.3.8" "google-genai==1.11.0" "evaluate==0.4.3" "rouge-score==0.1.2" "bert-score==0.3.13" "ragas==0.2.15" "langchain-openai==0.3.14"


In [ ]:
from __future__ import annotations

import ast
import json
import math
import os
import random
import re
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Iterable, Sequence

import numpy as np
import pandas as pd
from datasets import Dataset, load_dataset
from IPython.display import display
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATASET_ID = "SMARTICT/Pubmed-RAG-TR"
DATASET_REVISION = "7c6016f"
EVALUATION_SIZE = 600
LEGACY_TEST_FRACTION = 0.06
CHUNK_SIZE = 512
CHUNK_OVERLAP = 50
FINAL_K = 5
FIRST_STAGE_K = 5
HYBRID_WEIGHTS = (0.5, 0.5)
GENERATION_MODEL = "gemma-3-27b-it"
REQUEST_DELAY_SECONDS = 3.0

RUN_RAG = False
RUN_TEXT_METRICS = False
RUN_RAGAS = False
RUN_BEIR = False
SYSTEMS_TO_RUN = ["base", "v1", "v2", "v3", "no_reranker"]
RAG_LIMIT = EVALUATION_SIZE

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
HF_TOKEN = os.getenv("HF_TOKEN") or None

MODEL_IDS = {
    "base_embedding": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "domain_embedding": "SMARTICT/pubmedbert-base-embeddings-tr-pubmed-10k",
    "base_colbert": "colbert-ir/colbertv2.0",
    "reranker_v1": "SMARTICT/gte-multilingual-reranker-base-pubmed-tr-v1",
    "reranker_v2": "SMARTICT/gte-reranker-modernbert-base-pubmed-tr-v1",
}
MODEL_IDS

## 1. Load the paper evaluation set

In [ ]:
def parse_list(value: Any) -> list[str]:
    if isinstance(value, (list, tuple, np.ndarray)):
        parsed = list(value)
    elif value is None or (isinstance(value, float) and np.isnan(value)):
        parsed = []
    elif isinstance(value, str):
        try:
            parsed = json.loads(value)
        except json.JSONDecodeError:
            parsed = ast.literal_eval(value)
    else:
        raise TypeError(type(value))
    if not isinstance(parsed, list):
        raise ValueError("QA field did not decode to a list")
    return [str(item).strip() for item in parsed if str(item).strip()]

raw = load_dataset(
    DATASET_ID, revision=DATASET_REVISION, split="train", token=HF_TOKEN
)
legacy_test = raw.train_test_split(
    test_size=LEGACY_TEST_FRACTION, seed=SEED
)["test"]
paper_test = legacy_test.select(range(EVALUATION_SIZE))

rows = []
for record in paper_test:
    questions = parse_list(record["Questions"])
    answers = parse_list(record["Answer"])
    if not questions or len(questions) != len(answers):
        continue
    rows.append(
        {
            "id": record["id"],
            "pmid": record["PMID"],
            "context": record["tr_contents"],
            "question": questions[0],
            "reference_answer": answers[0],
        }
    )

evaluation_df = pd.DataFrame(rows)
assert len(raw) == 10_001 and len(legacy_test) == 601 and len(evaluation_df) == 600
evaluation_df.head(3)

## 2. Retrieval and fusion primitives


In [ ]:
def unicode_tokens(text: str) -> list[str]:
    return re.findall(r"\w+", text.lower(), flags=re.UNICODE)

def legacy_beir_tokens(text: str) -> list[str]:
    # Preserved only for comparison with the archived BEIR notebook.
    # It strips Turkish-specific characters because it accepts ASCII a-z only.
    return re.findall(r"[a-z0-9]+", text.lower())

def ranked_ids(scores: dict[str, float], limit: int | None = None) -> list[str]:
    ordered = sorted(scores, key=lambda doc_id: (-scores[doc_id], doc_id))
    return ordered if limit is None else ordered[:limit]

def weighted_rrf(
    rankings: Sequence[Sequence[str]],
    weights: Sequence[float] | None = None,
    offset: int = 60,
) -> dict[str, float]:
    if weights is None:
        weights = [1.0] * len(rankings)
    if len(rankings) != len(weights):
        raise ValueError("Each ranking requires one weight")
    fused: dict[str, float] = {}
    for ranking, weight in zip(rankings, weights):
        for rank, doc_id in enumerate(ranking, start=1):
            fused[doc_id] = fused.get(doc_id, 0.0) + weight / (offset + rank)
    return fused

def manuscript_rrf(*rankings: Sequence[str]) -> list[str]:
    scores = weighted_rrf(rankings, offset=0)
    return ranked_ids(scores)

# Unit checks: consensus should win; ties resolve by ID.
assert manuscript_rrf(["b", "a", "c"], ["a", "c", "b"])[0] == "a"
assert ranked_ids({"b": 1.0, "a": 1.0}) == ["a", "b"]
print("Retrieval/fusion unit checks passed.")

## 3. RAG configurations


In [ ]:
@dataclass(frozen=True)
class SystemSpec:
    label: str
    embedding_model: str
    hybrid: bool
    reranker_mode: str
    reranker_models: tuple[str, ...] = ()

SYSTEM_SPECS = {
    "base": SystemSpec(
        "Base RAG", MODEL_IDS["base_embedding"], False, "colbert",
        (MODEL_IDS["base_colbert"],),
    ),
    "v1": SystemSpec(
        "RAG V1", MODEL_IDS["domain_embedding"], True, "cross_encoder",
        (MODEL_IDS["reranker_v1"],),
    ),
    "v2": SystemSpec(
        "RAG V2", MODEL_IDS["domain_embedding"], True, "cross_encoder",
        (MODEL_IDS["reranker_v2"],),
    ),
    "v3": SystemSpec(
        "RAG V3", MODEL_IDS["domain_embedding"], True, "rrf",
        (MODEL_IDS["reranker_v1"], MODEL_IDS["reranker_v2"]),
    ),
    "no_reranker": SystemSpec(
        "RAG NO RERANKER", MODEL_IDS["domain_embedding"], True, "none"
    ),
}

pd.DataFrame([asdict(spec) | {"key": key} for key, spec in SYSTEM_SPECS.items()]).set_index("key")

In [ ]:
class RetrievalIndex:
    def __init__(self, documents: pd.DataFrame, embedding_model_id: str):
        from langchain_text_splitters import RecursiveCharacterTextSplitter
        from rank_bm25 import BM25Okapi
        from sentence_transformers import SentenceTransformer

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, length_function=len
        )
        chunks = []
        for row in documents.itertuples(index=False):
            for chunk_number, text in enumerate(splitter.split_text(row.context)):
                chunks.append(
                    {
                        "chunk_id": f"{row.id}::chunk-{chunk_number:04d}",
                        "document_id": row.id,
                        "text": text,
                    }
                )
        self.chunks = pd.DataFrame(chunks).set_index("chunk_id", drop=False)
        self.embedding_model = SentenceTransformer(embedding_model_id, token=HF_TOKEN)
        self.embeddings = self.embedding_model.encode(
            self.chunks["text"].tolist(),
            batch_size=32,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=True,
        )
        self.bm25 = BM25Okapi([unicode_tokens(text) for text in self.chunks["text"]])

    def dense(self, query: str, k: int) -> list[str]:
        query_vector = self.embedding_model.encode(
            [query], normalize_embeddings=True, convert_to_numpy=True
        )[0]
        scores = self.embeddings @ query_vector
        order = np.argsort(scores)[::-1][:k]
        return self.chunks.iloc[order]["chunk_id"].tolist()

    def lexical(self, query: str, k: int) -> list[str]:
        scores = self.bm25.get_scores(unicode_tokens(query))
        order = np.argsort(scores)[::-1][:k]
        return self.chunks.iloc[order]["chunk_id"].tolist()

    def retrieve(self, query: str, hybrid: bool, k: int) -> list[str]:
        dense = self.dense(query, k)
        if not hybrid:
            return dense
        lexical = self.lexical(query, k)
        fused = weighted_rrf([lexical, dense], weights=HYBRID_WEIGHTS, offset=60)
        return ranked_ids(fused)

    def texts(self, chunk_ids: Sequence[str]) -> list[str]:
        return self.chunks.loc[list(chunk_ids), "text"].tolist()

In [ ]:
RAG_PROMPT = """Bağlamda yer alan bilgileri kullanarak soruya kapsamlı bir cevap verin.
Sadece sorulan soruya yanıt verin; yanıt kısa ve soruyla ilgili olmalıdır.
İlgili olduğunda kaynak belgenin numarasını belirtin.
Eğer cevap bağlamdan çıkarılamıyorsa cevap vermeyin.

Bağlam:
{context}

Soru:
{question}

Cevap:
"""

class RAGRunner:
    def __init__(self, spec: SystemSpec, index: RetrievalIndex):
        self.spec = spec
        self.index = index
        self.rerankers: list[Any] = []
        if spec.reranker_mode in {"cross_encoder", "rrf"}:
            from sentence_transformers import CrossEncoder
            self.rerankers = [
                CrossEncoder(model_id, trust_remote_code=True, token=HF_TOKEN)
                for model_id in spec.reranker_models
            ]
        elif spec.reranker_mode == "colbert":
            try:
                from ragatouille import RAGPretrainedModel
            except ImportError as exc:
                raise ImportError(
                    "Base RAG requires `pip install ragatouille==0.0.9.post2`."
                ) from exc
            self.rerankers = [RAGPretrainedModel.from_pretrained(spec.reranker_models[0])]

    def rerank(self, query: str, candidate_ids: Sequence[str]) -> list[str]:
        if self.spec.reranker_mode == "none":
            return list(candidate_ids)[:FINAL_K]
        texts = self.index.texts(candidate_ids)
        if self.spec.reranker_mode == "colbert":
            ranked = self.rerankers[0].rerank(query=query, documents=texts, k=FINAL_K)
            return [candidate_ids[item["result_index"]] for item in ranked]

        pairs = [(query, text) for text in texts]
        rankings = []
        for model in self.rerankers:
            scores = np.asarray(model.predict(pairs, show_progress_bar=False)).reshape(-1)
            order = np.argsort(scores)[::-1]
            rankings.append([candidate_ids[index] for index in order])
        if self.spec.reranker_mode == "cross_encoder":
            return rankings[0][:FINAL_K]
        return manuscript_rrf(*rankings)[:FINAL_K]

    def retrieve(self, query: str) -> tuple[list[str], list[str]]:
        candidate_ids = self.index.retrieve(query, self.spec.hybrid, FIRST_STAGE_K)
        final_ids = self.rerank(query, candidate_ids)
        return final_ids, self.index.texts(final_ids)

def generate_answer(question: str, contexts: Sequence[str]) -> str:
    from google import genai
    from google.genai import types

    api_key = os.getenv("GOOGLE_API_KEY")
    if not api_key:
        raise RuntimeError("Set GOOGLE_API_KEY before enabling generation.")
    client = genai.Client(api_key=api_key)
    response = client.models.generate_content(
        model=GENERATION_MODEL,
        contents=[RAG_PROMPT.format(context="\n\n".join(contexts), question=question)],
        config=types.GenerateContentConfig(max_output_tokens=100, temperature=0.0),
    )
    return response.text.strip()

In [ ]:
def read_completed_ids(path: Path) -> set[str]:
    if not path.exists():
        return set()
    completed = set()
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                completed.add(json.loads(line)["id"])
    return completed

def run_rag_system(key: str, frame: pd.DataFrame, index_cache: dict[str, RetrievalIndex]) -> Path:
    spec = SYSTEM_SPECS[key]
    if spec.embedding_model not in index_cache:
        index_cache[spec.embedding_model] = RetrievalIndex(frame, spec.embedding_model)
    runner = RAGRunner(spec, index_cache[spec.embedding_model])
    output_path = OUTPUT_DIR / f"rag_{key}.jsonl"
    completed = read_completed_ids(output_path)

    with output_path.open("a", encoding="utf-8") as handle:
        for row in frame.head(RAG_LIMIT).itertuples(index=False):
            if row.id in completed:
                continue
            started = time.perf_counter()
            chunk_ids, contexts = runner.retrieve(row.question)
            answer = generate_answer(row.question, contexts)
            record = {
                "id": row.id,
                "question": row.question,
                "reference_answer": row.reference_answer,
                "source_context": row.context,
                "generated_answer": answer,
                "retrieved_chunk_ids": chunk_ids,
                "retrieved_contexts": contexts,
                "system": key,
                "system_spec": asdict(spec),
                "generation_model": GENERATION_MODEL,
                "elapsed_seconds": time.perf_counter() - started,
            }
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
            handle.flush()
            time.sleep(REQUEST_DELAY_SECONDS)
    return output_path

if RUN_RAG:
    indexes: dict[str, RetrievalIndex] = {}
    for system_key in SYSTEMS_TO_RUN:
        print(system_key, run_rag_system(system_key, evaluation_df, indexes))
else:
    print("RAG generation skipped. Set RUN_RAG=True and provide GOOGLE_API_KEY.")

## 4. ROUGE and multilingual BERTScore

In [ ]:
def load_rag_output(key: str) -> pd.DataFrame:
    path = OUTPUT_DIR / f"rag_{key}.jsonl"
    frame = pd.read_json(path, lines=True)
    if frame["id"].duplicated().any():
        raise ValueError(f"Duplicate IDs in {path}")
    expected = set(evaluation_df.head(RAG_LIMIT)["id"])
    if set(frame["id"]) != expected:
        missing = expected - set(frame["id"])
        raise ValueError(f"{path} is incomplete; {len(missing)} expected IDs are missing")
    return frame.sort_values("id").reset_index(drop=True)

def compute_text_metrics(frame: pd.DataFrame) -> dict[str, float]:
    import evaluate

    predictions = frame["generated_answer"].astype(str).tolist()
    references = frame["reference_answer"].astype(str).tolist()
    rouge = evaluate.load("rouge").compute(predictions=predictions, references=references)
    bert = evaluate.load("bertscore").compute(
        predictions=predictions, references=references,
        model_type="bert-base-multilingual-cased"
    )
    return {
        "rouge-1": float(rouge["rouge1"]),
        "rouge-2": float(rouge["rouge2"]),
        "rouge-L": float(rouge["rougeL"]),
        "bertscore-f1": float(np.mean(bert["f1"])),
        "bertscore-precision": float(np.mean(bert["precision"])),
        "bertscore-recall": float(np.mean(bert["recall"])),
    }

if RUN_TEXT_METRICS:
    text_metric_table = pd.DataFrame(
        {SYSTEM_SPECS[key].label: compute_text_metrics(load_rag_output(key))
         for key in SYSTEMS_TO_RUN}
    ).T
    display(text_metric_table)
else:
    print("Text metrics skipped. Enable after RAG JSONL files are complete.")

## 5. RAGAS metrics

In [ ]:
def compute_ragas(key: str, evaluator_model: str = "gpt-4o-mini") -> pd.DataFrame:
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Set OPENAI_API_KEY before running RAGAS.")

    from langchain_openai import ChatOpenAI
    from ragas import EvaluationDataset, SingleTurnSample, evaluate
    from ragas.llms import LangchainLLMWrapper
    from ragas.metrics import (
        Faithfulness,
        LLMContextPrecisionWithReference,
        LLMContextRecall,
        ResponseRelevancy,
    )

    frame = load_rag_output(key)
    samples = [
        SingleTurnSample(
            user_input=row.question,
            response=row.generated_answer,
            retrieved_contexts=list(row.retrieved_contexts),
            reference=row.reference_answer,
        )
        for row in frame.itertuples(index=False)
    ]
    dataset = EvaluationDataset(samples=samples)
    evaluator = LangchainLLMWrapper(ChatOpenAI(model=evaluator_model, temperature=0))
    metrics = [
        Faithfulness(llm=evaluator),
        ResponseRelevancy(llm=evaluator),
        LLMContextRecall(llm=evaluator),
        LLMContextPrecisionWithReference(llm=evaluator),
    ]
    scores = evaluate(dataset=dataset, metrics=metrics, llm=evaluator).to_pandas()
    scores.insert(0, "id", frame["id"].tolist())
    rename = {
        "response_relevancy": "answer_relevancy",
        "llm_context_recall": "context_recall",
        "llm_context_precision_with_reference": "context_precision",
    }
    scores = scores.rename(columns=rename)
    output = OUTPUT_DIR / f"ragas_{key}.csv"
    scores.to_csv(output, index=False)
    return scores

if RUN_RAGAS:
    for system_key in SYSTEMS_TO_RUN:
        display(compute_ragas(system_key).head())
else:
    print("RAGAS skipped. Set RUN_RAGAS=True only after complete RAG outputs exist.")

## 6. Paired significance tests

In [ ]:
RAGAS_COLUMNS = ["faithfulness", "answer_relevancy", "context_recall", "context_precision"]

def paired_wilcoxon_holm(
    base: pd.DataFrame, variant: pd.DataFrame, variant_name: str, alpha: float = 0.05
) -> pd.DataFrame:
    left = base[["id", *RAGAS_COLUMNS]].copy()
    right = variant[["id", *RAGAS_COLUMNS]].copy()
    if left["id"].duplicated().any() or right["id"].duplicated().any():
        raise ValueError("Paired tests require unique sample IDs.")
    paired = left.merge(right, on="id", suffixes=("_base", "_variant"), validate="one_to_one")
    if len(paired) != len(left) or len(paired) != len(right):
        raise ValueError("Base and variant score files do not contain identical IDs.")

    rows = []
    for metric in RAGAS_COLUMNS:
        clean = paired[[f"{metric}_base", f"{metric}_variant"]].dropna()
        base_values = clean.iloc[:, 0].to_numpy()
        variant_values = clean.iloc[:, 1].to_numpy()
        if np.allclose(variant_values - base_values, 0):
            statistic, p_value = 0.0, 1.0
        else:
            result = wilcoxon(
                variant_values, base_values, alternative="greater",
                zero_method="wilcox", method="auto"
            )
            statistic, p_value = float(result.statistic), float(result.pvalue)
        rows.append(
            {
                "system": variant_name, "metric": metric, "n": len(clean),
                "base_mean": base_values.mean(), "variant_mean": variant_values.mean(),
                "delta": variant_values.mean() - base_values.mean(),
                "statistic": statistic, "p_raw": p_value,
            }
        )
    table = pd.DataFrame(rows)
    reject, adjusted, _, _ = multipletests(table["p_raw"], alpha=alpha, method="holm")
    table["p_holm"] = adjusted
    table["significant"] = reject
    return table

if all((OUTPUT_DIR / f"ragas_{key}.csv").exists() for key in SYSTEMS_TO_RUN):
    base_scores = pd.read_csv(OUTPUT_DIR / "ragas_base.csv")
    significance = pd.concat(
        [
            paired_wilcoxon_holm(
                base_scores, pd.read_csv(OUTPUT_DIR / f"ragas_{key}.csv"),
                SYSTEM_SPECS[key].label
            )
            for key in ["no_reranker", "v1", "v2", "v3"]
        ],
        ignore_index=True,
    )
    display(significance)
else:
    print("Paired tests await complete per-sample RAGAS CSV files.")

## 7. Manuscript RAG results (archived reference)

In [ ]:
reported_rag_results = pd.DataFrame.from_dict(
    {
        "Base RAG": [0.892, 0.713, 0.850, 0.864, 0.482, 0.368, 0.455, 0.794, 0.763, 0.830],
        "RAG NO RERANKER": [0.930, 0.773, 0.939, 0.853, 0.518, 0.400, 0.488, 0.807, 0.773, 0.846],
        "RAG V1": [0.918, 0.762, 0.938, 0.918, 0.517, 0.401, 0.490, 0.807, 0.773, 0.846],
        "RAG V2": [0.907, 0.752, 0.930, 0.913, 0.523, 0.407, 0.498, 0.809, 0.776, 0.848],
        "RAG V3": [0.920, 0.762, 0.939, 0.910, 0.521, 0.406, 0.493, 0.809, 0.774, 0.849],
    },
    orient="index",
    columns=[
        "Faithfulness", "Answer Relevancy", "Context Recall", "Context Precision",
        "ROUGE-1", "ROUGE-2", "ROUGE-L", "BERTScore F1",
        "BERTScore Precision", "BERTScore Recall",
    ],
)
reported_rag_results["Average"] = reported_rag_results.mean(axis=1)
reported_rag_results

## 8. BEIR-style benchmark

In [ ]:
def build_benchmark(frame: pd.DataFrame):
    corpus = dict(zip(frame["id"], frame["context"]))
    queries = {f"q::{row.id}": row.question for row in frame.itertuples(index=False)}
    qrels = {f"q::{row.id}": {row.id: 1} for row in frame.itertuples(index=False)}
    return corpus, queries, qrels

def evaluate_at_k(
    qrels: dict[str, dict[str, int]], results: dict[str, dict[str, float]], k: int = 10
) -> dict[str, float]:
    ndcg_values, ap_values, recall_values, precision_values, rr_values = [], [], [], [], []
    for query_id, relevant in qrels.items():
        relevant_ids = {doc_id for doc_id, grade in relevant.items() if grade > 0}
        ranking = ranked_ids(results.get(query_id, {}), limit=k)
        gains = [relevant.get(doc_id, 0) for doc_id in ranking]
        dcg = sum((2**gain - 1) / math.log2(rank + 1) for rank, gain in enumerate(gains, start=1))
        ideal = sorted(relevant.values(), reverse=True)[:k]
        idcg = sum((2**gain - 1) / math.log2(rank + 1) for rank, gain in enumerate(ideal, start=1))
        ndcg_values.append(dcg / idcg if idcg else 0.0)

        hits = 0
        precision_sum = 0.0
        first_relevant_rank = None
        for rank, doc_id in enumerate(ranking, start=1):
            if doc_id in relevant_ids:
                hits += 1
                precision_sum += hits / rank
                if first_relevant_rank is None:
                    first_relevant_rank = rank
        denominator = min(len(relevant_ids), k)
        ap_values.append(precision_sum / denominator if denominator else 0.0)
        recall_values.append(hits / len(relevant_ids) if relevant_ids else 0.0)
        precision_values.append(hits / k)
        rr_values.append(1.0 / first_relevant_rank if first_relevant_rank else 0.0)

    return {
        f"NDCG@{k}": float(np.mean(ndcg_values)),
        f"MAP@{k}": float(np.mean(ap_values)),
        f"Recall@{k}": float(np.mean(recall_values)),
        f"P@{k}": float(np.mean(precision_values)),
        f"MRR@{k}": float(np.mean(rr_values)),
    }

def rrf_result_dict(*result_sets: dict[str, dict[str, float]]) -> dict[str, dict[str, float]]:
    query_ids = set().union(*(result.keys() for result in result_sets))
    return {
        query_id: weighted_rrf(
            [ranked_ids(result.get(query_id, {})) for result in result_sets], offset=0
        )
        for query_id in query_ids
    }

# Metric smoke test with one relevant document.
_smoke_qrels = {"q": {"d1": 1}}
_smoke_results = {"q": {"d2": 2.0, "d1": 1.0}}
_smoke = evaluate_at_k(_smoke_qrels, _smoke_results, k=2)
assert np.isclose(_smoke["MAP@2"], 0.5) and np.isclose(_smoke["MRR@2"], 0.5)
print("Benchmark metric unit check passed.")

In [ ]:
def run_beir_style_benchmark(frame: pd.DataFrame, candidate_k: int = 100) -> pd.DataFrame:
    from rank_bm25 import BM25Okapi
    from sentence_transformers import CrossEncoder, SentenceTransformer

    corpus, queries, qrels = build_benchmark(frame)
    document_ids = list(corpus)
    documents = [corpus[doc_id] for doc_id in document_ids]

    # BM25 first stage: exact archived tokenizer.
    bm25 = BM25Okapi([legacy_beir_tokens(text) for text in documents])
    bm25_results = {}
    for query_id, query in queries.items():
        scores = bm25.get_scores(legacy_beir_tokens(query))
        order = np.argsort(scores)[::-1][:candidate_k]
        bm25_results[query_id] = {document_ids[i]: float(scores[i]) for i in order}

    # Dense first stage: released PubMedBERT-TR embedding model.
    encoder = SentenceTransformer(MODEL_IDS["domain_embedding"], token=HF_TOKEN)
    document_vectors = encoder.encode(
        documents, batch_size=32, normalize_embeddings=True, convert_to_numpy=True,
        show_progress_bar=True
    )
    query_ids = list(queries)
    query_vectors = encoder.encode(
        [queries[qid] for qid in query_ids], batch_size=32, normalize_embeddings=True,
        convert_to_numpy=True, show_progress_bar=True
    )
    dense_results = {}
    for query_id, vector in zip(query_ids, query_vectors):
        scores = document_vectors @ vector
        order = np.argsort(scores)[::-1][:candidate_k]
        dense_results[query_id] = {document_ids[i]: float(scores[i]) for i in order}

    reranker_v1 = CrossEncoder(MODEL_IDS["reranker_v1"], trust_remote_code=True, token=HF_TOKEN)
    reranker_v2 = CrossEncoder(MODEL_IDS["reranker_v2"], trust_remote_code=True, token=HF_TOKEN)

    def rerank(first_stage: dict[str, dict[str, float]], model: Any) -> dict[str, dict[str, float]]:
        output = {}
        for query_id, query in queries.items():
            candidates = ranked_ids(first_stage[query_id], limit=candidate_k)
            scores = np.asarray(
                model.predict([(query, corpus[doc_id]) for doc_id in candidates],
                              batch_size=64, show_progress_bar=False)
            ).reshape(-1)
            output[query_id] = {doc_id: float(score) for doc_id, score in zip(candidates, scores)}
        return output

    dense_v1 = rerank(dense_results, reranker_v1)
    dense_v2 = rerank(dense_results, reranker_v2)
    bm25_v1 = rerank(bm25_results, reranker_v1)
    bm25_v2 = rerank(bm25_results, reranker_v2)

    systems = {
        "BM25 (Baseline)": bm25_results,
        "PubMedBERT-Emb (Dense Baseline)": dense_results,
        "Dense + gte-multilingual-reranker-base-pubmed-tr": dense_v1,
        "Dense + gte-reranker-modernbert-base-pubmed-tr": dense_v2,
        "Dense + RRF (both rerankers)": rrf_result_dict(dense_v1, dense_v2),
        "BM25 + gte-multilingual-reranker-base-pubmed-tr": bm25_v1,
        "BM25 + gte-reranker-modernbert-base-pubmed-tr": bm25_v2,
        "BM25 + RRF (both rerankers)": rrf_result_dict(bm25_v1, bm25_v2),
    }
    table = pd.DataFrame(
        {name: evaluate_at_k(qrels, results, k=10) for name, results in systems.items()}
    ).T
    table.index.name = "Model"
    return table

if RUN_BEIR:
    fresh_beir_results = run_beir_style_benchmark(evaluation_df)
    fresh_beir_results.to_csv(OUTPUT_DIR / "beir_results.csv")
    display(fresh_beir_results)
else:
    print("Full BEIR-style benchmark skipped. Set RUN_BEIR=True on a CUDA runtime.")

## 9. Manuscript BEIR results and a consistency diagnostic

In [ ]:
reported_beir_results = pd.DataFrame.from_dict(
    {
        "BM25 (Baseline)": [0.9403, 0.9317, 0.9676, 0.0968, 0.9317],
        "PubMedBERT-Emb (Dense Baseline)": [0.7752, 0.7502, 0.8536, 0.0854, 0.7502],
        "Dense + gte-multilingual-reranker-base-pubmed-tr": [0.3855, 0.2473, 0.8536, 0.0854, 0.7502],
        "Dense + gte-reranker-modernbert-base-pubmed-tr": [0.8476, 0.8456, 0.8536, 0.0854, 0.8456],
        "Dense + RRF (both rerankers)": [0.8330, 0.8258, 0.8536, 0.0854, 0.8238],
        "BM25 + gte-multilingual-reranker-base-pubmed-tr": [0.0498, 0.0319, 0.1106, 0.0111, 0.9317],
        "BM25 + gte-reranker-modernbert-base-pubmed-tr": [0.9514, 0.9437, 0.9759, 0.0976, 0.9437],
        "BM25 + RRF (both rerankers)": [0.9601, 0.9522, 0.9842, 0.0984, 0.9514],
    },
    orient="index",
    columns=["NDCG@10", "MAP@10", "Recall@10", "P@10", "MRR@10"],
)
reported_beir_results.index.name = "Model"
reported_beir_results["single-qrel MAP=MRR check"] = np.isclose(
    reported_beir_results["MAP@10"], reported_beir_results["MRR@10"], atol=5e-5
)
reported_beir_results